#### ***05 — LSTM***

PyTorch LSTM forecaster on the same daily series, evaluated with the same rolling-origin protocol used in `03` and `04`. The model is **multivariate** — input features are `energy_kwh` + `temp_mean` + `is_weekend` + `is_holiday` + `month` — and forecasts the 7-day horizon **autoregressively** at inference time.

Training (sweep + early stopping) is GPU-bound, so it is extracted to a script. The notebook only consumes the cached weights and runs evaluation in a few seconds.

#### ***Train via script first (one-off, GPU recommended)***

> Heavy step extracted to a script to keep this notebook short and to avoid
> kernel hangs. Run once from the project root:

```bash
python scripts/lstm_train.py
```

What it does:
- Sweeps `seq_len ∈ {14, 28}`, `hidden_size ∈ {32, 64}`, `num_layers ∈ {1, 2}` (8 configs).
- Holds out the last **60 days of train** as validation.
- Early-stopping on validation loss (`patience = 12`).
- Persists weights + normalisation stats + sweep table to `data/lstm/`.

Auto-detects `cuda → mps → cpu`. On the RTX 5070 Ti the full sweep finishes in
**~5–10 min**; on M1 Pro `mps` you can expect **15–25 min**.

#### ***Imports***

In [ ]:
import sys
import json
from pathlib import Path
sys.path.insert(0, '..')   # make `scripts/` importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from scripts.lstm_train import LSTMForecaster, get_device, FEATURE_COLS, TARGET_COL
from scripts.evaluation import evaluate_model

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True

#### ***Load processed data and split***

In [ ]:
data = pd.read_csv('../data/processed_daily.csv', parse_dates=['date'], index_col='date')

TEST_DAYS = 30
HORIZON   = 7
SEASON    = 7

split_date = data.index.max() - pd.Timedelta(days=TEST_DAYS - 1)
train_y    = data.loc[data.index <  split_date, 'energy_kwh']
test_y     = data.loc[data.index >= split_date, 'energy_kwh']

print(f'Train: {len(train_y)}   Test: {len(test_y)}')

#### ***Load the trained model and sweep results***

In [ ]:
LSTM_DIR = Path('../data/lstm')

# Sweep table (one row per configuration tried)
sweep = pd.read_csv(LSTM_DIR / 'sweep_results.csv').sort_values('best_val_loss')
sweep

In [ ]:
# Winner configuration and normalisation stats
with open(LSTM_DIR / 'best_config.json') as f:
    best = json.load(f)

print(f'Winner: {best["name"]}   val_loss={best["val_loss"]:.4f}')
print(f'Config: {best["config"]}')

#### ***Rebuild the model and load the cached weights***

In [ ]:
device = get_device()
print(f'Device: {device}')

cfg   = best['config']
model = LSTMForecaster(
    n_features  = len(FEATURE_COLS),
    hidden_size = cfg['hidden_size'],
    num_layers  = cfg['num_layers'],
    dropout     = cfg['dropout'],
).to(device)

state = torch.load(LSTM_DIR / 'best_model.pt', map_location=device)
model.load_state_dict(state)
model.eval()

# Quick sanity check: number of parameters
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

#### ***Training curve of the winning configuration***

In [ ]:
history = pd.read_csv(LSTM_DIR / 'best_history.csv')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history['epoch'], history['train_loss'], label='Train loss')
ax.plot(history['epoch'], history['val_loss'],   label='Val loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (normalised target)')
ax.set_title(f'Training curve — {best["name"]}')
ax.legend(); plt.tight_layout(); plt.show()

#### ***Inference helper — autoregressive multi-step forecasting***

The model is trained one-step-ahead. To produce a 7-day forecast we roll the
input window forward, feeding back each predicted value as the next step's
target feature while keeping the (known) exogenous future values fixed.

In [ ]:
SEQ_LEN     = cfg['seq_len']
X_mean      = np.asarray(best['norms']['X_mean'], dtype=np.float32)
X_std       = np.asarray(best['norms']['X_std'],  dtype=np.float32)
Y_MEAN      = best['norms']['y_mean']
Y_STD       = best['norms']['y_std']
TARGET_IDX  = FEATURE_COLS.index(TARGET_COL)

def lstm_forecast_fn(history, horizon):
    last_date    = history.index[-1]
    past_dates   = pd.date_range(end=last_date, periods=SEQ_LEN, freq='D')
    future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=horizon, freq='D')

    # Past window (energy + exog), normalised
    past_features = data.loc[past_dates,   FEATURE_COLS].values.astype(np.float32)
    future_exog   = data.loc[future_dates, FEATURE_COLS].values.astype(np.float32)
    window = (past_features - X_mean) / X_std

    preds = []
    with torch.no_grad():
        for i in range(horizon):
            x       = torch.from_numpy(window[np.newaxis]).to(device)
            y_norm  = model(x).item()
            y_pred  = y_norm * Y_STD + Y_MEAN
            preds.append(y_pred)

            # Roll the window: drop oldest step, append (predicted energy, known future exog)
            next_feat              = future_exog[i].copy()
            next_feat[TARGET_IDX]  = y_pred
            next_feat_n            = (next_feat - X_mean) / X_std
            window                 = np.concatenate([window[1:], next_feat_n[np.newaxis]], axis=0)

    return np.asarray(preds, dtype=float)

#### ***Rolling-origin evaluation — same windows as 03 / 04***

In [ ]:
windows_l, per_l, sum_l = evaluate_model(lstm_forecast_fn, train_y, test_y,
                                          horizon=HORIZON, season=SEASON)

print('Per-window metrics:')
print(per_l.round(3).to_string(index=False))

#### ***Full leaderboard — baselines + SARIMA + SARIMAX + LSTM***

We re-run every previous model with the same evaluator so the comparison is end-to-end identical. SARIMA / SARIMAX use the winning orders cached in `data/sarima_grid_results.csv`.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings; warnings.filterwarnings('ignore')

# --- Baselines
def baseline_historical_mean(h, k): return np.full(k, h.mean())
def baseline_last_value(h, k):       return np.full(k, h.iloc[-1])
def baseline_seasonal_naive(h, k):
    last = h.iloc[-SEASON:].values
    return np.tile(last, int(np.ceil(k / SEASON)))[:k]
def baseline_drift(h, k):
    T = len(h)
    return h.iloc[-1] + (h.iloc[-1] - h.iloc[0]) / (T - 1) * np.arange(1, k + 1)

# --- SARIMA / SARIMAX from the cached grid
EXOG_COLS = ['temp_mean', 'is_weekend', 'is_holiday']
exog_full = data[EXOG_COLS]
grid = pd.read_csv('../data/sarima_grid_results.csv')

def pick_winner(grid_df, variant):
    sub = grid_df[grid_df['variant'] == variant].dropna(subset=['aic']).sort_values('aic').iloc[0]
    return (int(sub.p), int(sub.d), int(sub.q)), (int(sub.P), int(sub.D), int(sub.Q), int(sub.m))

order_s,  sorder_s  = pick_winner(grid, 'no_exog')
order_sx, sorder_sx = pick_winner(grid, 'exog')

def make_sarima_forecast_fn(order, sorder, exog_full=None):
    def forecast_fn(history, horizon):
        ex_train = exog_full.loc[history.index] if exog_full is not None else None
        future_idx = pd.date_range(history.index[-1] + pd.Timedelta(days=1),
                                   periods=horizon, freq='D')
        ex_future = exog_full.loc[future_idx] if exog_full is not None else None
        m = SARIMAX(history, exog=ex_train, order=order, seasonal_order=sorder,
                    enforce_stationarity=False, enforce_invertibility=False
                   ).fit(disp=False, maxiter=200)
        return m.forecast(steps=horizon, exog=ex_future).values
    return forecast_fn

sarima_fn  = make_sarima_forecast_fn(order_s,  sorder_s,  exog_full=None)
sarimax_fn = make_sarima_forecast_fn(order_sx, sorder_sx, exog_full=exog_full)

models = {
    'Historical mean':                  baseline_historical_mean,
    'Last value':                       baseline_last_value,
    'Seasonal naive':                   baseline_seasonal_naive,
    'Drift':                            baseline_drift,
    f'SARIMA {order_s}x{sorder_s}':     sarima_fn,
    f'SARIMAX {order_sx}x{sorder_sx}':  sarimax_fn,
    f'LSTM {best["name"]}':             lstm_forecast_fn,
}

summaries = {}
windows_by_model = {}
for name, fn in models.items():
    w, _, summary = evaluate_model(fn, train_y, test_y, horizon=HORIZON, season=SEASON)
    summaries[name] = summary
    windows_by_model[name] = w

summary_df = pd.DataFrame(summaries).T

def fmt(row, m):
    return f'{row[f"{m}_mean"]:.2f} ± {row[f"{m}_std"]:.2f}'

leaderboard = pd.DataFrame({
    'MAE':   summary_df.apply(lambda r: fmt(r, 'MAE'),   axis=1),
    'RMSE':  summary_df.apply(lambda r: fmt(r, 'RMSE'),  axis=1),
    'MAPE':  summary_df.apply(lambda r: f'{r["MAPE_mean"]*100:5.1f}% ± {r["MAPE_std"]*100:4.1f}%',   axis=1),
    'sMAPE': summary_df.apply(lambda r: f'{r["sMAPE_mean"]*100:5.1f}% ± {r["sMAPE_std"]*100:4.1f}%', axis=1),
    'MASE':  summary_df.apply(lambda r: fmt(r, 'MASE'),  axis=1),
    'Bias':  summary_df.apply(lambda r: f'{r["Bias_mean"]:+.2f}',  axis=1),
    'R²':    summary_df.apply(lambda r: f'{r["R2_mean"]:+.2f}',    axis=1),
})
leaderboard['_mase'] = summary_df['MASE_mean']
leaderboard = leaderboard.sort_values('_mase').drop(columns='_mase')
leaderboard

#### ***Forecasts vs actuals — LSTM, SARIMAX, seasonal naive***

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test_y.index, test_y.values, 'b-', lw=1.8, label='Actual')

for name, color in [(f'SARIMAX {order_sx}x{sorder_sx}', 'g'),
                    (f'LSTM {best["name"]}',            'orange'),
                    ('Seasonal naive',                  'magenta')]:
    idx   = np.concatenate([w['index']       for w in windows_by_model[name]])
    preds = np.concatenate([w['predictions'] for w in windows_by_model[name]])
    ls = ':' if name == 'Seasonal naive' else '--'
    ax.plot(idx, preds, ls, color=color, lw=1.2, label=name)

for w in windows_by_model['Seasonal naive']:
    ax.axvline(w['index'][0], color='grey', alpha=0.3, lw=0.8)

ax.set_xlabel('Date'); ax.set_ylabel('Energy (kWh)')
ax.set_title('Rolling-origin forecasts — LSTM vs SARIMAX vs seasonal naive')
ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

#### ***Summary***

- The LSTM is trained one-step-ahead and forecasts the 7-day horizon autoregressively.
- All seven models in the leaderboard share the same train / test split, the same rolling-origin evaluator (`scripts/evaluation.py`), and the same metrics (`scripts/metrics.py`) — the comparison is byte-for-byte fair.
- Look at **MASE** for cross-model ranking and at **Bias** for systematic over- / under-prediction.

**Next:** `06_timesfm.ipynb` — zero-shot foundation model from Google. No training; just inference on the same 7-day windows. Closes the comparison.